In [1]:
##### Bibliotecas
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# Ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss

# Optuna
import optuna

# Organização do dataset
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract = True

In [2]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val', 'test']}

/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [3]:
# Extração de features + Congelamento dos parâmetros
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False
            
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.densenet201(pretrained=False)
set_parameter_requires_grad(model, feature_extract)

model_densenet = "/home/jovyan/models/densenet201-model-95.pth"
state_dict = torch.load(model_densenet)

del state_dict['classifier.weight']
del state_dict['classifier.bias']

model.load_state_dict(state_dict, strict=False)

num_features = model.classifier.in_features
model.to(device)  

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [4]:
# Função de treinamento e validação
def train_step(engine, batch):
    x, y = batch
    x, y = x.to(device), y.to(device)

    model.train()
    y_pred = model(x)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        x, y = batch
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        return y_pred, y

# Função `objective` para busca bayesiana com Optuna
def objective(trial):
    global model, optimizer, criterion
    
    # Hiperparâmetros
    dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
    dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
    num_neurons_fc1 = trial.suggest_categorical("num_neurons_fc1", [256, 512, 1024])
    num_neurons_fc2 = trial.suggest_categorical("num_neurons_fc2", [128, 256, 512])
    activation = trial.suggest_categorical("activation", ["ReLU"])
    batch_size = trial.suggest_categorical("batch_size", [128])
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam"])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    momentum = trial.suggest_uniform("momentum", 0.7, 0.99) if optimizer_name == "SGD" else None

    # Função de ativação
    activation_fn = getattr(nn, activation)()

    # Redefinir o DataLoader com o batch size sugerido
    dataloaders_dict = {
        'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
        'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    }

    # Configurar o modelo com os hiperparâmetros sugeridos
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate1),
        nn.Linear(num_features, num_neurons_fc1),
        activation_fn,
        nn.Dropout(p=dropout_rate2),
        nn.Linear(num_neurons_fc1, num_neurons_fc2),
        activation_fn,
        nn.Linear(num_neurons_fc2, 2)
    )
    model.to(device)

    # Configurar o otimizador
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "SGD":
        optimizer = optim.SGD(params_to_update, lr=lr, momentum=momentum)
    else:
        optimizer = optim.Adam(params_to_update, lr=lr)

    # Definir função de perda
    criterion = nn.CrossEntropyLoss()

    # Ignite trainers
    trainer = Engine(train_step)
    evaluator = Engine(validation_step)

    # Métricas
    val_metrics = {
        "accuracy": Accuracy(),
        "loss": Loss(criterion)
    }
    for name, metric in val_metrics.items():
        metric.attach(evaluator, name)

    @trainer.on(Events.EPOCH_COMPLETED)
    def log_training_results(engine):
        evaluator.run(dataloaders_dict['val'])
        metrics = evaluator.state.metrics
        print(f"Val Accuracy: {metrics['accuracy']:.4f}")
        
        # Early Stopping
        score_function = lambda engine: engine.state.metrics['accuracy']
        handler = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, handler)

    # Executar o treinamento
    trainer.run(dataloaders_dict['train'], max_epochs=100)

    # Obter a acurácia final
    evaluator.run(dataloaders_dict['val'])
    return evaluator.state.metrics["accuracy"]
# Rodar o estudo Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Rodar o estudo Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Melhor conjunto de hiperparâmetros
print("Best trial:")
trial = study.best_trial
print(f"Accuracy: {trial.value}")
print("Best hyperparameters: ", trial.params)


with open("result_dense_net.txt", 'a', encoding='utf-8') as file:
    file.write(f'Best trial --- \n Accuracy: {trial.value}\n \n \n')
    file.write(f'Best hyperparameters : {trial.params}')

[I 2025-02-06 23:23:59,602] A new study created in memory with name: no-name-4b9173b0-20eb-4526-ac6d-cbc3413f623a
/tmp/ipykernel_238/2656494870.py:29: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
/tmp/ipykernel_238/2656494870.py:30: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
/tmp/ipykernel_238/2656494870.py:36: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)


Val Accuracy: 0.5714
Val Accuracy: 0.7179
Val Accuracy: 0.9011
Val Accuracy: 0.9158
Val Accuracy: 0.9341
Val Accuracy: 0.9341
Val Accuracy: 0.9414
Val Accuracy: 0.9524
Val Accuracy: 0.9451
Val Accuracy: 0.9634
Val Accuracy: 0.9560
Val Accuracy: 0.9560
Val Accuracy: 0.9560
Val Accuracy: 0.9634
Val Accuracy: 0.9560
Val Accuracy: 0.9634
Val Accuracy: 0.9560
Val Accuracy: 0.9634
Val Accuracy: 0.9634


2025-02-07 00:19:19,684 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,685 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,685 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,685 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,685 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,686 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,686 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,686 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:19,687 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Val Accuracy: 0.9634


2025-02-07 00:19:48,477 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,477 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,478 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,478 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,478 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,478 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,479 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,479 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:19:48,480 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
[I 2025-02-07 00:19:48,480] Trial 0 finished with value: 0.96336

...

Val Accuracy: 0.6410
Val Accuracy: 0.5971
Val Accuracy: 0.5531
Val Accuracy: 0.5495
Val Accuracy: 0.5934
Val Accuracy: 0.6813
Val Accuracy: 0.7143
Val Accuracy: 0.8022
Val Accuracy: 0.8462
Val Accuracy: 0.8535
Val Accuracy: 0.8901
Val Accuracy: 0.9121
Val Accuracy: 0.9121
Val Accuracy: 0.9048
Val Accuracy: 0.9121
Val Accuracy: 0.9304
Val Accuracy: 0.9304
Val Accuracy: 0.9341
Val Accuracy: 0.9341
Val Accuracy: 0.9377
Val Accuracy: 0.9487
Val Accuracy: 0.9414
Val Accuracy: 0.9487
Val Accuracy: 0.9524
Val Accuracy: 0.9560
Val Accuracy: 0.9560
Val Accuracy: 0.9560
Val Accuracy: 0.9560
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9634
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9597
Val Accuracy: 0.9634
Val Accuracy: 0.9597
Val Accuracy: 0.9634


2025-02-12 19:42:15,544 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,544 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,545 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,545 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,546 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,546 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,546 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,546 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,546 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:15,547 ignite.handlers.early_stopping.EarlyStop

Val Accuracy: 0.9597


2025-02-12 19:42:44,043 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,044 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,044 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,044 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,045 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,045 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,045 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,045 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,045 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-12 19:42:44,046 ignite.handlers.early_stopping.EarlyStop

Best trial:
Accuracy: 0.989010989010989
Best hyperparameters:  {'dropout1': 0.3372612925395379, 'dropout2': 0.2672714454014481, 'num_neurons_fc1': 512, 'num_neurons_fc2': 256, 'activation': 'ReLU', 'batch_size': 128, 'optimizer': 'Adam', 'lr': 0.0006562369354118999}
